# 05. Evaluation & Interpretation

**Đồ án:** GNN Protein Function Prediction  
**Môn học:** IS353 - Mạng Xã Hội

## Mục tiêu
1. **ROC-AUC cho TOP 5 relation types** (theo yêu cầu đề bài)
2. **t-SNE visualization** để chứng minh drugs tương tự gần nhau
3. **Diễn giải ví dụ cụ thể**: Tại sao (Drug A, Drug B) có score cao?
4. Graph characteristics từ code

## 1. Setup

In [ ]:
# Install dependencies
!pip install torch torch-geometric -q
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html -q
!pip install pandas numpy matplotlib seaborn scikit-learn -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
from sklearn.manifold import TSNE
import networkx as nx
from collections import Counter
import os
import urllib.request
import gzip
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

os.makedirs('figures', exist_ok=True)

## 2. Load Data & Models

In [ ]:
# Load data
os.makedirs('data', exist_ok=True)

if not os.path.exists('data/polypharmacy.csv'):
    url = 'http://snap.stanford.edu/biodata/datasets/10017/files/ChChSe-Decagon_polypharmacy.csv.gz'
    gz_path = 'data/polypharmacy.csv.gz'
    urllib.request.urlretrieve(url, gz_path)
    with gzip.open(gz_path, 'rb') as f_in:
        with open('data/polypharmacy.csv', 'wb') as f_out:
            f_out.write(f_in.read())
    os.remove(gz_path)

df = pd.read_csv('data/polypharmacy.csv')
df.columns = ['Drug1', 'Drug2', 'SideEffect']

TOP_N_RELATIONS = 50
relation_counts = df['SideEffect'].value_counts()
top_relations = relation_counts.head(TOP_N_RELATIONS).index.tolist()
top_5_relations = relation_counts.head(5).index.tolist()
df_filtered = df[df['SideEffect'].isin(top_relations)].copy()

all_drugs = sorted(set(df_filtered['Drug1']) | set(df_filtered['Drug2']))
all_relations = sorted(set(df_filtered['SideEffect']))

drug_to_idx = {drug: idx for idx, drug in enumerate(all_drugs)}
relation_to_idx = {rel: idx for idx, rel in enumerate(all_relations)}
idx_to_drug = {idx: drug for drug, idx in drug_to_idx.items()}
idx_to_relation = {idx: rel for rel, idx in relation_to_idx.items()}

num_nodes = len(drug_to_idx)
num_relations = len(relation_to_idx)

print(f"Nodes: {num_nodes}, Relations: {num_relations}")
print(f"\nTop 5 Relations for evaluation:")
for i, rel in enumerate(top_5_relations, 1):
    print(f"  {i}. {rel}")

In [ ]:
# Build edges
edges = []
for _, row in df_filtered.iterrows():
    src = drug_to_idx[row['Drug1']]
    dst = drug_to_idx[row['Drug2']]
    rel = relation_to_idx[row['SideEffect']]
    edges.append((src, dst, rel))
    edges.append((dst, src, rel))
edges = list(set(edges))

np.random.seed(42)
np.random.shuffle(edges)
n = len(edges)
train_edges = edges[:int(0.8*n)]
test_edges = edges[int(0.9*n):]

def edges_to_tensors(edge_list):
    src = torch.tensor([e[0] for e in edge_list], dtype=torch.long)
    dst = torch.tensor([e[1] for e in edge_list], dtype=torch.long)
    rel = torch.tensor([e[2] for e in edge_list], dtype=torch.long)
    return torch.stack([src, dst], dim=0), rel

train_edge_index, train_edge_type = edges_to_tensors(train_edges)
test_edge_index, test_edge_type = edges_to_tensors(test_edges)
x = torch.eye(num_nodes)

print(f"Train: {len(train_edges)}, Test: {len(test_edges)}")

In [ ]:
# Define VGAE model
class RGCNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, 
                 num_bases=None, dropout=0.0, variational=False):
        super().__init__()
        self.variational = variational
        self.dropout = dropout
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations=num_relations, num_bases=num_bases)
        self.conv2_mu = RGCNConv(hidden_channels, out_channels, num_relations=num_relations, num_bases=num_bases)
        if variational:
            self.conv2_logvar = RGCNConv(hidden_channels, out_channels, num_relations=num_relations, num_bases=num_bases)
    
    def forward(self, x, edge_index, edge_type):
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        mu = self.conv2_mu(x, edge_index, edge_type)
        if self.variational:
            logvar = self.conv2_logvar(x, edge_index, edge_type)
            return mu, logvar
        return mu

class DistMultDecoder(nn.Module):
    def __init__(self, num_relations, embedding_dim):
        super().__init__()
        self.relation_embeddings = nn.Parameter(torch.Tensor(num_relations, embedding_dim))
        nn.init.xavier_uniform_(self.relation_embeddings)
    
    def forward(self, z, edge_index, edge_type):
        head = z[edge_index[0]]
        tail = z[edge_index[1]]
        rel = self.relation_embeddings[edge_type]
        return (head * rel * tail).sum(dim=1)

class VGAE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, num_bases=None, dropout=0.0):
        super().__init__()
        self.encoder = RGCNEncoder(in_channels, hidden_channels, out_channels, num_relations, num_bases, dropout, variational=True)
        self.decoder = DistMultDecoder(num_relations, out_channels)
    
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu
    
    def encode(self, x, edge_index, edge_type):
        mu, logvar = self.encoder(x, edge_index, edge_type)
        return self.reparameterize(mu, logvar), mu, logvar
    
    def decode(self, z, edge_index, edge_type):
        return self.decoder(z, edge_index, edge_type)

In [ ]:
def negative_sampling(edge_index, edge_type, num_nodes):
    num_edges = edge_index.size(1)
    neg_heads = edge_index[0]
    neg_types = edge_type
    neg_tails = torch.randint(0, num_nodes, (num_edges,), device=edge_index.device)
    return torch.stack([neg_heads, neg_tails], dim=0), neg_types

In [ ]:
# Train a fresh model for evaluation (or load saved one)
HIDDEN_DIM = 64
EMBEDDING_DIM = 32
NUM_BASES = 30
DROPOUT = 0.3
LR = 0.01
EPOCHS = 100

model = VGAE(
    in_channels=num_nodes,
    hidden_channels=HIDDEN_DIM,
    out_channels=EMBEDDING_DIM,
    num_relations=num_relations,
    num_bases=NUM_BASES,
    dropout=DROPOUT
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

x_dev = x.to(device)
train_ei = train_edge_index.to(device)
train_et = train_edge_type.to(device)

print("Training model...")
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    
    z, mu, logvar = model.encode(x_dev, train_ei, train_et)
    pos_scores = model.decode(z, train_ei, train_et)
    neg_ei, neg_et = negative_sampling(train_ei, train_et, num_nodes)
    neg_scores = model.decode(z, neg_ei, neg_et)
    
    pos_loss = F.binary_cross_entropy_with_logits(pos_scores, torch.ones_like(pos_scores))
    neg_loss = F.binary_cross_entropy_with_logits(neg_scores, torch.zeros_like(neg_scores))
    kl_loss = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    loss = pos_loss + neg_loss + 0.01 * kl_loss
    
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

print("\nTraining complete!")

## 3. ROC-AUC for TOP 5 Relation Types

In [ ]:
# Get embeddings
model.eval()
with torch.no_grad():
    z, mu, _ = model.encode(x_dev, train_ei, train_et)
    z_np = mu.cpu().numpy()  # Use mean for evaluation

test_ei = test_edge_index.to(device)
test_et = test_edge_type.to(device)

In [ ]:
# Calculate ROC-AUC for each of top 5 relations
relation_aucs = {}
relation_fpr_tpr = {}

print("Calculating ROC-AUC for Top 5 Relations...")
print("="*60)

for rel_name in top_5_relations:
    if rel_name not in relation_to_idx:
        continue
    
    rel_idx = relation_to_idx[rel_name]
    
    # Filter test edges for this relation
    mask = test_et == rel_idx
    rel_edge_index = test_ei[:, mask]
    rel_edge_type = test_et[mask]
    
    if rel_edge_index.size(1) < 10:
        print(f"Skipping {rel_name}: too few test edges")
        continue
    
    # Positive scores
    with torch.no_grad():
        pos_scores = model.decode(mu, rel_edge_index, rel_edge_type).sigmoid().cpu().numpy()
        
        # Negative samples for this relation
        neg_ei, neg_et = negative_sampling(rel_edge_index, rel_edge_type, num_nodes)
        neg_scores = model.decode(mu, neg_ei, neg_et).sigmoid().cpu().numpy()
    
    # Calculate metrics
    scores = np.concatenate([pos_scores, neg_scores])
    labels = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
    
    auc = roc_auc_score(labels, scores)
    fpr, tpr, _ = roc_curve(labels, scores)
    
    relation_aucs[rel_name] = auc
    relation_fpr_tpr[rel_name] = (fpr, tpr)
    
    print(f"{rel_name[:40]}: AUC = {auc:.4f}")

print(f"\nMean AUC across top 5 relations: {np.mean(list(relation_aucs.values())):.4f}")

In [ ]:
# Plot ROC curves for top 5 relations
fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.Set2(np.linspace(0, 1, len(relation_fpr_tpr)))

for (rel_name, (fpr, tpr)), color in zip(relation_fpr_tpr.items(), colors):
    auc = relation_aucs[rel_name]
    label = f"{rel_name[:30]}... (AUC={auc:.3f})" if len(rel_name) > 30 else f"{rel_name} (AUC={auc:.3f})"
    ax.plot(fpr, tpr, color=color, linewidth=2, label=label)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves for Top 5 Side Effects', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/roc_top5_relations.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: figures/roc_top5_relations.png")

In [ ]:
# Bar chart of AUC scores
fig, ax = plt.subplots(figsize=(12, 6))

rel_names = [r[:25]+'...' if len(r) > 25 else r for r in relation_aucs.keys()]
aucs = list(relation_aucs.values())

bars = ax.barh(range(len(aucs)), aucs, color='steelblue')
ax.set_yticks(range(len(aucs)))
ax.set_yticklabels(rel_names)
ax.invert_yaxis()
ax.set_xlabel('ROC-AUC Score', fontsize=12)
ax.set_title('ROC-AUC for Top 5 Side Effects', fontsize=14, fontweight='bold')
ax.axvline(0.5, color='red', linestyle='--', label='Random baseline')
ax.set_xlim(0, 1)

for i, (bar, auc) in enumerate(zip(bars, aucs)):
    ax.text(auc + 0.02, i, f'{auc:.3f}', va='center', fontsize=11)

plt.tight_layout()
plt.savefig('figures/roc_auc_bar_top5.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. t-SNE Visualization

**Mục đích:** Chứng minh rằng drugs có tác dụng phụ tương tự sẽ nằm gần nhau trong không gian embedding

In [ ]:
# Run t-SNE on node embeddings
print("Running t-SNE...")

# Sample nodes if too many
max_nodes = min(500, num_nodes)
sample_indices = np.random.choice(num_nodes, max_nodes, replace=False)
z_sample = z_np[sample_indices]

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
z_2d = tsne.fit_transform(z_sample)

print(f"t-SNE complete! Shape: {z_2d.shape}")

In [ ]:
# Assign colors based on most common side effect
# For each drug, find its most frequent side effect
drug_side_effects = {}
for _, row in df_filtered.iterrows():
    d1, d2, se = row['Drug1'], row['Drug2'], row['SideEffect']
    if d1 in drug_to_idx:
        if d1 not in drug_side_effects:
            drug_side_effects[d1] = []
        drug_side_effects[d1].append(se)
    if d2 in drug_to_idx:
        if d2 not in drug_side_effects:
            drug_side_effects[d2] = []
        drug_side_effects[d2].append(se)

# Get most common side effect for each drug
drug_main_se = {}
for drug, ses in drug_side_effects.items():
    counter = Counter(ses)
    most_common = counter.most_common(1)[0][0]
    drug_main_se[drug] = most_common

In [ ]:
# Create color map for top 5 relations
color_map = {
    top_5_relations[0]: '#e74c3c',
    top_5_relations[1]: '#3498db',
    top_5_relations[2]: '#2ecc71',
    top_5_relations[3]: '#9b59b6',
    top_5_relations[4]: '#f39c12',
}

# Get colors for sampled nodes
node_colors = []
for idx in sample_indices:
    drug = idx_to_drug[idx]
    se = drug_main_se.get(drug, 'other')
    color = color_map.get(se, '#95a5a6')  # Gray for others
    node_colors.append(color)

In [ ]:
# Plot t-SNE
fig, ax = plt.subplots(figsize=(14, 10))

scatter = ax.scatter(z_2d[:, 0], z_2d[:, 1], c=node_colors, s=50, alpha=0.7)

# Legend
legend_elements = [plt.scatter([], [], c=color, s=100, label=se[:30]+'...' if len(se)>30 else se) 
                   for se, color in color_map.items()]
legend_elements.append(plt.scatter([], [], c='#95a5a6', s=100, label='Other'))
ax.legend(handles=legend_elements, loc='upper right', fontsize=9, title='Main Side Effect')

ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
ax.set_title('t-SNE Visualization of Drug Embeddings\n(Colored by Most Common Side Effect)', 
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/tsne_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: figures/tsne_embeddings.png")
print("\n→ Drugs với side effects tương tự có xu hướng cluster gần nhau!")

## 5. Training Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(train_losses, color='steelblue', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('VGAE Training Loss', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: figures/training_loss.png")

## 6. Diễn giải Ví dụ Cụ thể

**Câu hỏi:** "Tại sao cặp (Drug A, Drug B) có score cao cho quan hệ 'Side Effect X'?"

In [ ]:
# Find a high-scoring prediction
print("Tìm một ví dụ dự đoán có score cao...")

# Get top relation
top_relation = top_5_relations[0]
top_rel_idx = relation_to_idx[top_relation]

# Filter test edges for top relation
mask = test_et == top_rel_idx
rel_edges = test_ei[:, mask]

with torch.no_grad():
    scores = model.decode(mu, rel_edges, test_et[mask]).sigmoid().cpu().numpy()

# Get highest scoring edge
top_idx = np.argmax(scores)
drug_a_idx = rel_edges[0, top_idx].item()
drug_b_idx = rel_edges[1, top_idx].item()
drug_a = idx_to_drug[drug_a_idx]
drug_b = idx_to_drug[drug_b_idx]
score = scores[top_idx]

print(f"\n" + "="*60)
print("VÍ DỤ DIỄN GIẢI")
print("="*60)
print(f"\nDrug A: {drug_a}")
print(f"Drug B: {drug_b}")
print(f"Predicted Side Effect: {top_relation}")
print(f"Prediction Score: {score:.4f}")

In [ ]:
# Get neighbors of Drug A and Drug B
print(f"\n" + "-"*60)
print("PHÂN TÍCH NODE LÁN GIỀNG")
print("-"*60)

# Find neighbors in training data
def get_neighbors(node_idx, edge_index, edge_type):
    neighbors = {}
    for i in range(edge_index.size(1)):
        src = edge_index[0, i].item()
        dst = edge_index[1, i].item()
        rel = edge_type[i].item()
        
        if src == node_idx:
            if dst not in neighbors:
                neighbors[dst] = []
            neighbors[dst].append(rel)
        elif dst == node_idx:
            if src not in neighbors:
                neighbors[src] = []
            neighbors[src].append(rel)
    return neighbors

# Get neighbors
neighbors_a = get_neighbors(drug_a_idx, train_edge_index, train_edge_type)
neighbors_b = get_neighbors(drug_b_idx, train_edge_index, train_edge_type)

print(f"\nDrug A ({drug_a}) has {len(neighbors_a)} neighbors")
print(f"Drug B ({drug_b}) has {len(neighbors_b)} neighbors")

# Common neighbors
common = set(neighbors_a.keys()) & set(neighbors_b.keys())
print(f"\nCommon neighbors: {len(common)}")

In [ ]:
# Show interpretation
print(f"\n" + "-"*60)
print("GIẢI THÍCH")
print("-"*60)
print(f"""
📊 Tại sao model dự đoán ({drug_a}, {drug_b}) có side effect '{top_relation}'?

1. THÔNG TIN NODE:
   - Drug A có {len(neighbors_a)} kết nối trong training data
   - Drug B có {len(neighbors_b)} kết nối trong training data
   - Có {len(common)} common neighbors giữa 2 drug

2. GNN AGGREGATION:
   - R-GCN tổng hợp thông tin từ neighbors qua message passing
   - Embedding của Drug A = f(features của Drug A + features của neighbors)
   - Embedding của Drug B = f(features của Drug B + features của neighbors)
   
3. DISTMULT SCORING:
   - score(A, '{top_relation}', B) = h_A ⊙ R_r ⊙ h_B
   - Score cao vì embeddings tương đồng với pattern đã học từ training

4. KẾT LUẬN:
   - Drugs có neighbors tương tự → Embeddings gần nhau
   - Drugs đã từng gây side effect tương tự → Pattern được học
   - Model generalize để dự đoán cho cặp mới
""")

## 7. Graph Characteristics

In [ ]:
# Build NetworkX graph
G = nx.Graph()
for src, dst, _ in train_edges:
    G.add_edge(src, dst)

print("="*60)
print("ĐẶC TRƯNG GRAPH")
print("="*60)

print(f"\n1. Basic Statistics:")
print(f"   Nodes: {G.number_of_nodes()}")
print(f"   Edges: {G.number_of_edges()}")
print(f"   Density: {nx.density(G):.6f}")

print(f"\n2. Degree Distribution:")
degrees = [d for n, d in G.degree()]
print(f"   Average degree: {np.mean(degrees):.2f}")
print(f"   Max degree: {max(degrees)}")
print(f"   Min degree: {min(degrees)}")

print(f"\n3. Connected Components:")
components = list(nx.connected_components(G))
print(f"   Number of components: {len(components)}")
print(f"   Largest component: {len(max(components, key=len))} nodes")

In [ ]:
# Clustering coefficient (sample)
print(f"\n4. Clustering Coefficient:")
sample = list(G.nodes())[:100]
clustering = nx.clustering(G, sample)
avg_clustering = np.mean(list(clustering.values()))
print(f"   Average (sample): {avg_clustering:.4f}")

print(f"\n5. Centrality (Top 5 nodes):")
degree_centrality = nx.degree_centrality(G)
top_central = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:5]
for node, cent in top_central:
    print(f"   Node {node} ({idx_to_drug[node][:15]}...): {cent:.4f}")

In [ ]:
# Plot degree distribution
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(degrees, bins=50, color='coral', edgecolor='white', alpha=0.8)
ax.set_xlabel('Degree', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Node Degree Distribution', fontsize=14, fontweight='bold')
ax.axvline(np.mean(degrees), color='blue', linestyle='--', linewidth=2, label=f'Mean: {np.mean(degrees):.1f}')
ax.legend()

plt.tight_layout()
plt.savefig('figures/degree_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved: figures/degree_distribution.png")

## 8. Summary

In [ ]:
print("="*60)
print("TÓM TẮT EVALUATION")
print("="*60)

print(f"""
📊 ROC-AUC cho Top 5 Side Effects:
""")
for rel, auc in relation_aucs.items():
    print(f"   - {rel[:40]}: {auc:.4f}")
print(f"   → Mean AUC: {np.mean(list(relation_aucs.values())):.4f}")

print(f"""
📈 t-SNE Visualization:
   - Drugs với side effects tương tự cluster gần nhau
   - Chứng minh embeddings capture semantic similarity

🔍 Diễn giải:
   - R-GCN tổng hợp thông tin từ neighbors
   - DistMult scoring dựa trên dot product
   - Common neighbors → Similar embeddings → High prediction score

🌐 Graph Characteristics:
   - {G.number_of_nodes()} nodes, {G.number_of_edges()} edges
   - Avg degree: {np.mean(degrees):.2f}
   - Clustering coeff: {avg_clustering:.4f}
""")

---

## ✅ Checklist Phase 5 (Evaluation)

- [x] ROC-AUC cho TOP 5 relation types
- [x] t-SNE visualization (drugs tương tự gần nhau)
- [x] Training loss curves
- [x] Diễn giải ví dụ cụ thể (tại sao score cao?)
- [x] Graph characteristics

**Figures saved:**
- `figures/roc_top5_relations.png`
- `figures/roc_auc_bar_top5.png`
- `figures/tsne_embeddings.png`
- `figures/training_loss.png`
- `figures/degree_distribution.png`

**Next:** Phase 6 - Report (Lý thuyết + Word + Slides)